# Cifar-100 Classification with basic layers

When we first execute the following cell it will download the data, which takes some minutes (should be <5min)

In [ ]:
import tensorflow as tf
import numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
print(f"Training data {x_train.shape}")
print(f"Training label {y_train.shape}")
print(f"Test data {x_test.shape}")
print(f"Test label {y_test.shape}")
print(f"Max number of training label {np.max(y_train)}")


## Look at some examples

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 3, figsize=(6, 4)) # create subplots 2 rows 3 columnns

for index, img in enumerate(x_train[0:6]):
    panel = axs[int(index/3),index%3]   # find the panel by  2dimensional indexing
    panel.imshow(img, cmap="gray_r")    # plot the picture in the panel


In [ ]:
# define the names of the labels
# Name of all classes in CIFAR-100
classes = ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy',
           'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 
           'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 
           'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 
           'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 
           'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 
           'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 
           'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 
           'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 
           'wolf', 'woman', 'worm']
print(len(classes)) # should be 100

In [ ]:
labels = y_train[0:6].reshape(2,3);
print(labels)
np.array(classes)[labels]

## Network without Convolutions

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, BatchNormalization, Dropout, InputLayer
from tensorflow.keras.regularizers import l2

opt=Adam(
    learning_rate=0.0001,
    beta_1=0.8,
    beta_2=0.999)

model = Sequential([
      InputLayer((32,32,3)),   
      Flatten(),               # 32x32x3 -> 3084
      BatchNormalization(),
      Dense(512, activation='sigmoid', kernel_regularizer=l2(0.01)),
      Dropout(0.2),
      BatchNormalization(),
      Dense(512, activation='sigmoid', kernel_regularizer=l2(0.01)),
      Dropout(0.2),
      Dense(100, activation="softmax", kernel_regularizer=l2(0.01))
      ])
model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"]) 
model.summary()

In [ ]:
# this will run for about 7s per epoch without GPU
model.fit(
    x_train, 
    y_train, 
    validation_data=(x_test, y_test),
    epochs=25, 
    batch_size=300
)

In [ ]:
model.predict(x_test)

In [ ]:
prediction = tf.argmax(model.predict(x_test),axis=1)
print(prediction)

In [ ]:
y_test.shape  # this is ground truth

In [ ]:
np.sum(y_test.reshape(-1)==prediction)

## Plot Confusion Matrix

In [ ]:
#!pip install scikit-learn

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(prediction, y_test.reshape(-1))
display(cm)

In [ ]:
# Plot the confusion matrix
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(24,24))
ax = fig.add_subplot(211)
cax = ax.matshow(cm)
plt.title('Confusion matrix of the classifier')
fig.colorbar(cax)
# Set ticks positions
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation='vertical', fontsize=7)
ax.set_yticklabels(classes, fontsize=7)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

Scores in Tabular form

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test.reshape(-1), prediction, target_names=classes, digits=3))